## Graph rag application

### Setup Config


In [ ]:
# configure these from env or config
DOC_INTEL_ENDPOINT = "" ## Document intelligence endpoint, e.g., "https://<your-resource-name>.cognitiveservices.azure.com/" for OCR
DOC_INTEL_KEY = ""

AZ_OPENAI_ENDPOINT =  "" ## Azure OpenAI endpoint, e.g., "https://<your-resource-name>.openai.azure.com/"
AZ_OPENAI_KEY = "" ## Azure OpenAI key
# model name you deployed, e.g., "gpt-4o-mini" or whatever in your subscription
AZ_OPENAI_MODEL = "gpt-4.1"
EMBEDDING_MODEL = "text-embedding-3-large"


POSTGRES_CONN = {
    "dbname": "mygraphdb",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost",
    "port": 5432
}



### Step 1 Read documents


In [2]:
# file: ingest.py
import os
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential
from pdfminer.high_level import extract_text



document_client = DocumentIntelligenceClient(
    endpoint=DOC_INTEL_ENDPOINT,
    credential=AzureKeyCredential(DOC_INTEL_KEY)
)

def extract_text_from_pdf(pdf_path: str):
    with open(pdf_path, "rb") as f:
        poller = document_client.begin_analyze_document(
            model_id="prebuilt-read",
            body=f  # <-- correct parameter name
        )
        result = poller.result()

    pages = []
    for page in result.pages:
        lines = [line.content for line in page.lines]
        page_text = "\n".join(lines)
        pages.append({"page": page.page_number, "text": page_text})

    return pages




### Step 2 Gather Relationships

In [ ]:
 prompt = f"""
    Extract entities and relationships from the following text:

    {text}

    Respond ONLY with valid JSON in the format:
    {{
      "entities": [
        {{ "id": "...", "type": "...", "name": "...", "aliases": [] }}
      ],
      "relationships": [
        {{ "source": "...", "target": "...", "type": "..." }}
      ]
    }}
    """

In [4]:
RELATIONS_PROMPT = """
You are an expert in historical infrastructure, canal engineering, and 
document analysis. Your task is to extract a **clean, normalized, 
semantically rich knowledge graph** from the following document text.

-------------------------------------
Document:
{TEXT}
-------------------------------------

## EXTRACTION RULES

### 1. ENTITY TYPES (choose best fit)
- Canal
- Lock
- Bridge
- GuardGate
- Dam
- Feeder
- Structure
- Worker
- Place
- Equipment
- Organization
- Event
- Issue
- Measurement
- Vessel
- Waterway

### 2. ALIAS DETECTION (VERY IMPORTANT)
Unify references like:
- “E17”, “E-17”, “Lock E17”, “Lock 17” → SAME node
- “Old Erie”, “Ol Erie”, “Erie Canal”, “E Canal” → SAME node
- “Feeder Canal”, “Glens Falls Feeder” → SAME node

### 3. RELATIONSHIPS (meaningful only)
Use ONLY these types:
- `alias_of`
- `part_of`          (structure → canal)
- `located_in`       (structure → town/region)
- `maintained_by`
- `inspected_by`
- `damage_event`
- `repair_event`
- `mentions`
- `refers_to`
- `built_in`
- `constructed_by`
- `operational_status`
- `equipment_issue`

### 4. AVOID USELESS RELATIONSHIPS
DO NOT:
- connect everything to a document  
- create abstract “file” or “document” nodes  
- connect every sentence as a separate graph  

### 5. NORMALIZATION
Unify entity names into canonical form:
- Use common canal naming conventions  
- Use contextual reasoning  
- Use historical knowledge  

### 6. OUTPUT FORMAT
Return ONLY JSON in this shape:

{
  "entities": [
    {
      "id": "canonical_name",
      "type": "EntityType",
      "aliases": ["name1","name2"],
      "example_mention": "text snippet"
    }
  ],
  "relationships": [
    {
      "source": "canonical_name",
      "target": "canonical_name",
      "type": "relation_type",
      "evidence": "text snippet",
      "confidence": 0.95
    }
  ]
}

### 7. QUALITY REQUIREMENTS (CRITICAL)
- You MUST merge duplicates
- You MUST detect abbreviations and code names
- You MUST produce meaningful multi-hop graph structure
- You MUST infer correct relationships using context
- You MUST output only valid JSON

Begin now.
"""


In [5]:
SECOND_PASS_PROMPT = """
You are improving a partially extracted knowledge graph.

Task:
1. Merge nodes that represent the same concept.
2. Deduplicate aliases.
3. Improve entity categorization.
4. Add missing relationships implied by context.
5. Normalize names (e.g., Lock E17, E17 → Lock 17).
6. Strengthen graph structure by adding part_of relationships.

Return final graph JSON only.
"""

def refine_graph(graph_json, client, model="gpt-4.1"):
    prompt = SECOND_PASS_PROMPT + "\n\nRAW GRAPH:\n" + json.dumps(graph_json, indent=2)

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    raw = response.choices[0].message.content

    try:
        start = raw.index("{")
        end = raw.rindex("}") + 1
        return json.loads(raw[start:end])
    except:
        print("Refinement JSON parse error")
        print(raw)
        return graph_json


In [25]:
# file: extract_relations.py
import os
from openai import AzureOpenAI
import json

relation_client = AzureOpenAI(
    api_key=AZ_OPENAI_KEY,
    api_version="2024-05-01-preview",
    azure_endpoint=AZ_OPENAI_ENDPOINT
)

DEPLOYMENT = AZ_OPENAI_MODEL  # your GPT-4.1 deployment name

def extract_relationships(text, model="gpt-4.1"):
    prompt = RELATIONS_PROMPT.replace("{TEXT}", text)

    response = relation_client.chat.completions.create(
        model=AZ_OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    raw = response.choices[0].message.content

    # Some responses may wrap JSON -> extract safely
    try:
        start = raw.index("{")
        end = raw.rindex("}") + 1
        json_block = raw[start:end]
        return json.loads(json_block)
    except:
        print("Extraction error: Could not parse JSON")
        print(raw)
        return None





In [11]:
## Debug Extraction

if __name__ == "__main__":
    sample = "Worked on Erie canal today, E17 was jammed so Ol Erie was offline."
    data = extract_relationships(sample)
    print(json.dumps(data, indent=2))

{
  "entities": [
    {
      "id": "Erie Canal",
      "type": "Canal",
      "aliases": [
        "Old Erie",
        "Ol Erie",
        "E Canal",
        "Erie Canal"
      ],
      "example_mention": "Ol Erie was offline"
    },
    {
      "id": "Lock E17",
      "type": "Lock",
      "aliases": [
        "E17",
        "E-17",
        "Lock 17",
        "Lock E17"
      ],
      "example_mention": "E17 was jammed"
    },
    {
      "id": "Issue: Lock E17 Jammed",
      "type": "Issue",
      "aliases": [],
      "example_mention": "E17 was jammed"
    }
  ],
  "relationships": [
    {
      "source": "Lock E17",
      "target": "Erie Canal",
      "type": "part_of",
      "evidence": "Worked on Erie canal today, E17 was jammed",
      "confidence": 0.98
    },
    {
      "source": "Lock E17",
      "target": "Issue: Lock E17 Jammed",
      "type": "equipment_issue",
      "evidence": "E17 was jammed",
      "confidence": 0.97
    },
    {
      "source": "Erie Canal",
      "t

### Step 3 insert into Postgres

In [12]:
import os
import psycopg2
from psycopg2.extras import RealDictCursor
import json




GRAPH_NAME = "canals_graph"

def get_conn():
    return psycopg2.connect(**POSTGRES_CONN)

def ensure_age_graph():
    with get_conn() as conn:
        with conn.cursor() as cur:
            # Install AGE
            cur.execute("CREATE EXTENSION IF NOT EXISTS age;")
            # Load AGE into this session
            cur.execute("LOAD 'age';")
            # Required so `cypher()` is found
            cur.execute("SET search_path = ag_catalog, \"$user\", public;")
            # Create graph if missing
            cur.execute(f"""
                SELECT create_graph('{GRAPH_NAME}')
                WHERE NOT EXISTS (
                    SELECT 1 FROM ag_catalog.ag_graph WHERE name = '{GRAPH_NAME}'
                );
            """)
            conn.commit()
    print(f"✔ AGE graph '{GRAPH_NAME}' is ready.")

ensure_age_graph()


def escape(s):
    return s.replace('"', '\\"') if isinstance(s, str) else s

def upsert_node_and_aliases(node, tx_cursor):
    canonical = escape(node.get("name") or node.get("id"))
    labels    = node.get("type", "Entity")
    aliases   = [escape(a) for a in node.get("aliases", [])]
    example   = escape(node.get("example_mention", ""))

    # Convert list to AGTYPE literal
    aliases_literal = "[" + ", ".join(f'"{a}"' for a in aliases) + "]"

    cypher = f"""
        SELECT * FROM cypher('{GRAPH_NAME}', $$
            MATCH (n:{labels} {{canonical: "{canonical}"}})
            RETURN id(n)

            UNION

            CREATE (n:{labels} {{
                canonical: "{canonical}",
                aliases: {aliases_literal},
                example: "{example}"
            }})
            RETURN id(n)
        $$) AS (node_id agtype);
    """

    tx_cursor.execute(cypher)



def upsert_edge(edge, tx_cursor):
    src       = escape(edge.get("source") or edge.get("from"))
    tgt       = escape(edge.get("target") or edge.get("to"))
    rtype     = edge.get("type") or edge.get("relation")
    evidence  = escape(edge.get("evidence", ""))
    confidence = edge.get("confidence", 0)

    cypher = f"""
        SELECT * FROM cypher('{GRAPH_NAME}', $$
            MATCH (a {{canonical: "{src}"}})
            MATCH (b {{canonical: "{tgt}"}})

            OPTIONAL MATCH (a)-[existing:{rtype}]->(b)
            WITH a, b, existing
            WHERE existing IS NOT NULL
            RETURN id(existing)

            UNION

            CREATE (a)-[r:{rtype} {{
                evidence: "{evidence}",
                confidence: {confidence}
            }}]->(b)
            RETURN id(r)
        $$) AS (edge_id agtype);
    """

    tx_cursor.execute(cypher)


def to_agtype(params: dict) -> str:
    """Convert a Python dict into AGTYPE map syntax."""
    items = []
    for k, v in params.items():
        if v is None:
            items.append(f"{k}: null")
        elif isinstance(v, str):
            items.append(f'{k}: "{v}"')
        elif isinstance(v, (int, float)):
            items.append(f"{k}: {v}")
        elif isinstance(v, list):
            # stringify list with quotes
            lst = ", ".join([f'"{x}"' for x in v])
            items.append(f"{k}: [{lst}]")
        else:
            # fallback as string
            items.append(f'{k}: "{str(v)}"')
    return "{" + ", ".join(items) + "}"



def ingest_extractions(graph_json):
    conn = get_conn()
    try:
        with conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("LOAD 'age';")
                cur.execute("SET search_path = ag_catalog, \"$user\", public;")
                # Insert nodes
                for entity in graph_json["entities"]:
                    upsert_node_and_aliases(entity, cur)

                # Insert edges
                for rel in graph_json["relationships"]:
                    upsert_edge(rel, cur)

        print("✔ Graph updated.")
    finally:
        conn.close()


✔ AGE graph 'canals_graph' is ready.


### Step 4 Visualize Graph

In [24]:
!pip install pyvis


#### Visualization Helper


In [13]:
from pyvis.network import Network
import json

def parse_agtype_vertex(text):
    """Strip ::vertex and load JSON."""
    clean = text.replace("::vertex", "").strip()
    return json.loads(clean)

def parse_agtype_edge(text):
    """Strip ::edge and load JSON."""
    clean = text.replace("::edge", "").strip()
    return json.loads(clean)

def fetch_nodes():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("LOAD 'age';")
    cur.execute("SET search_path = ag_catalog, \"$user\", public;")

    cur.execute("""
    SELECT * FROM cypher('canals_graph',
        $$ MATCH (n) RETURN n $$)
    AS (n agtype);
    """)

    nodes = [parse_agtype_vertex(row[0]) for row in cur.fetchall()]
    conn.close()
    return nodes

def fetch_edges():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("LOAD 'age';")
    cur.execute("SET search_path = ag_catalog, \"$user\", public;")

    cur.execute("""
    SELECT * FROM cypher('canals_graph',
        $$ MATCH ()-[r]->() RETURN r $$)
    AS (r agtype);
    """)

    edges = [parse_agtype_edge(row[0]) for row in cur.fetchall()]
    conn.close()
    return edges


#### Build Visualizations

In [29]:
import json

def export_graph_for_cytoscape():
    nodes = fetch_nodes()
    edges = fetch_edges()

    cy_nodes = []
    cy_edges = []

    for n in nodes:
        node_id = n["id"]
        label   = n["label"]
        canon   = n["properties"].get("canonical","")
        aliases = n["properties"].get("aliases", [])

        cy_nodes.append({
            "data": {
                "id": str(node_id),
                "label": canon if canon else label,
                "type": label,
                "aliases": ", ".join(aliases)
            }
        })

    for e in edges:
        src = e["start_id"]
        tgt = e["end_id"]
        rel = e["label"]

        cy_edges.append({
            "data": {
                "id": f"{src}-{tgt}-{rel}",
                "source": str(src),
                "target": str(tgt),
                "label": rel
            }
        })

    return cy_nodes, cy_edges


In [30]:
def create_cytoscape_html(output="cyto_graph.html"):
    cy_nodes, cy_edges = export_graph_for_cytoscape()

    html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <title>Cytoscape Graph</title>

    <!-- Cytoscape.js -->
    <script src="https://cdnjs.cloudflare.com/ajax/libs/cytoscape/3.26.0/cytoscape.min.js"></script>

    <style>
        body {{
            margin: 0;
            overflow: hidden;
            background: #222;
        }}
        #cy {{
            width: 100vw;
            height: 100vh;
            display: block;
        }}
    </style>
</head>

<body>
<div id="cy"></div>

<script>
    var cy = cytoscape({{
        container: document.getElementById('cy'),

        elements: {{
            nodes: {json.dumps(cy_nodes)},
            edges: {json.dumps(cy_edges)}
        }},

        style: [
            {{
                selector: 'node',
                style: {{
                    'background-color': '#4da6ff',
                    'label': 'data(label)',
                    'color': 'white',
                    'font-size': 10,
                    'text-valign': 'center',
                    'text-halign': 'center'
                }}
            }},
            {{
                selector: 'node[type = "HistoricDistrict"]',
                style: {{
                    'background-color': '#ffd24d'
                }}
            }},
            {{
                selector: 'edge',
                style: {{
                    'width': 2,
                    'line-color': '#888',
                    'target-arrow-color': '#888',
                    'target-arrow-shape': 'triangle',
                    'curve-style': 'bezier',
                    'label': 'data(label)',
                    'font-size': 8,
                    'color': '#ccc'
                }}
            }}
        ],

        layout: {{
            name: 'cose',
            animate: true,
            fit: true,
            padding: 50
        }}
    }});
</script>

</body>
</html>
"""

    with open(output, "w", encoding="utf-8") as f:
        f.write(html_template)

    print(f"✔ Cytoscape graph written to: {output}")
    print("👉 Open it in your browser for a beautiful, smooth graph!")


In [39]:
create_cytoscape_html()

✔ Cytoscape graph written to: cyto_graph.html
👉 Open it in your browser for a beautiful, smooth graph!


### Step 5 Chat with Data

In [15]:
import psycopg2
from psycopg2.extensions import register_adapter, AsIs

def adapt_agtype(py_obj):
    # convert Python dict to AGTYPE map
    import json
    return AsIs("'" + json.dumps(py_obj) + "'::agtype")

register_adapter(dict, adapt_agtype)


In [16]:
from openai import AzureOpenAI
import os
import json

aoai = AzureOpenAI(
    api_key=AZ_OPENAI_KEY,
    azure_endpoint=AZ_OPENAI_ENDPOINT,
    api_version="2024-10-21",
)

GPT_MODEL = "gpt-4.1"   # your model name
GRAPH_NAME = "canals_graph"

def generate_cypher(question: str):
    prompt = f"""
You translate user questions into valid Apache AGE Cypher.

STRICT RULES:
- Return ONLY Cypher code.
- ONE LINE ONLY. No line breaks.
- No backticks, no markdown, no comments.
- No semicolons at the end.
- Must use the graph name: {GRAPH_NAME}
- Use ONLY: MATCH, WHERE, RETURN, OPTIONAL MATCH.
- Do NOT use CREATE, MERGE, DELETE, DROP, CALL, INSERT, SET.
- All strings must use double quotes.

SCHEMA HINTS:
- Nodes always have property: canonical
- Nodes may also have: aliases (list)
- Relationships can be any type, but you can match with `-[r]->`

User question:
\"\"\"{question}\"\"\"
"""

    response = aoai.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    cypher = response.choices[0].message.content.strip()

    # Force one-line format (just to be sure)
    cypher = " ".join(cypher.split())

    return cypher




### get data for chat

In [17]:
def run_cypher(cypher_query: str):
    cypher_query = cypher_query.strip().strip("`").strip(";")
    cypher_block = f"$${cypher_query}$$"

    conn = get_conn()
    cur = conn.cursor()

    cur.execute("LOAD 'age';")
    cur.execute("""SET search_path = ag_catalog, "$user", public;""")

    # IMPORTANT: use 2 argument version ONLY
    sql = f"""
        SELECT * FROM cypher(
            %s,
            {cypher_block}
        ) AS (result agtype);
    """

    cur.execute(sql, (GRAPH_NAME,))
    rows = cur.fetchall()
    conn.close()
    return [row[0] for row in rows]



In [18]:
def validate_cypher(cypher: str):
    if "delete" in cypher.lower():
        raise ValueError("Dangerous Cypher detected (DELETE). Aborting.")
    if "create graph" in cypher.lower():
        raise ValueError("Dangerous Cypher detected (CREATE GRAPH). Aborting.")
    if "drop" in cypher.lower():
        raise ValueError("Dangerous Cypher detected (DROP). Aborting.")


### get answers

In [60]:
def answer_from_results(question: str, results):
    prompt = f"""
You are summarizing graph query results from a historic canal knowledge graph.

User asked:
\"\"\"{question}\"\"\"

Here are the raw graph results (AGTYPE JSON-like):
{json.dumps(results, indent=2)}

Write a clear natural language answer for the user.
"""

    response = aoai.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )

    # FIXED: Use attribute access, not indexing
    return response.choices[0].message.content


In [44]:
def chat_with_graph(question: str):
    print("📌 Generating Cypher query...")
    cypher = generate_cypher(question)
    print("\nGenerated Cypher:")
    print(cypher)

    print("\n📌 Validating Cypher...")
    validate_cypher(cypher)
    print("\n✅ Cypher validated.")

    print("\n📌 Running Cypher on graph...")
    results = run_cypher(cypher)
    print("\nRaw Graph Results:")
    print(results)

    print("\n📌 Generating natural language answer...")
    answer = answer_from_results(question, results)

    return answer


In [66]:
chat_with_graph("Where is Gaurd Gate 3")
# chat_with_graph("What maintenance issues were mentioned on the Erie Canal?")
# chat_with_graph("List all alternative names for the Champlain Canal.")





📌 Generating Cypher query...



Generated Cypher:
MATCH (n) WHERE n.canonical = "Gaurd Gate 3" RETURN n

📌 Validating Cypher...

✅ Cypher validated.

📌 Running Cypher on graph...

Raw Graph Results:
[]

📌 Generating natural language answer...


'There is no information available about the location of Guard Gate 3 in the knowledge graph.'

### Main App

In [71]:
run_cypher("MATCH (n) RETURN n.canonical")


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [19]:
# === MAIN PIPELINE EXECUTION ===

import os
import json
from pathlib import Path

# Folder containing historical PDFs
DOCS_FOLDER = "./files"  # update if needed


import json
from pathlib import Path

def process_document(pdf_path: str):
    print(f"\n📄 Processing: {pdf_path}")

    pages = extract_text_from_pdf(pdf_path)

    # Convert pages to single text block
    if isinstance(pages, list):
        text = "\n".join([p["text"] for p in pages if p.get("text")])
    else:
        text = pages or ""

    if "This PDF file is protected" in text:
        print("❌ PDF is protected — cannot process.")
        return None

    print("\n---- EXTRACTED TEXT SAMPLE ----")
    print(text[:500])
    print("--------------------------------")

    if not text.strip():
        print("❌ No text extracted. Skipping.")
        return None

    # Extract relationships using LLM
    graph_json = extract_relationships(text)

    if not graph_json:
        print("❌ No relationships extracted.")
        return None

    # Save JSON to a file
    output_path = Path(pdf_path).with_suffix(".graph.json")
    with open(output_path, "w") as f:
        json.dump(graph_json, f, indent=2)

    print(f"💾 Saved graph JSON → {output_path}")
    return output_path







### load json helper

In [20]:
def ingest_json_file(json_path):
    import json

    

    with open(json_path, "r") as f:
        graph_json = json.load(f)

    ingest_extractions(graph_json)
    print(f"✔ Inserted {json_path} into PostgreSQL")



def parse_agtype_vertex(agtype_text):
    """
    Convert AGE vertex text to a Python dict.
    Example input:
    '{"id":123, "label":"TestNode", "properties":{"canonical":"hello"}}::vertex'
    """
    clean = agtype_text.replace("::vertex", "").strip()
    return json.loads(clean)


### Run main, execute Graph Building

In [42]:
# === Run for all PDFs in folder ===

results = []

# for file in Path(DOCS_FOLDER).glob("*.pdf"):
#     result = process_document(str(file))
#     if result:
#         results.append(result)


# pdf_path = "C:\\repos\\postgres_ai\\ai_test-db\\python\\files\\02a_Features-ChamplainCanal_print.pdf"

# pdf_path = "C:\\repos\\postgres_ai\\ai_test-db\\python\\files\\02b_Features-ErieCanal_print.pdf"

pdf_path = "C:\\repos\postgres_ai\\ai_test-db\python\\files\\1905 Whitford History_of_the_Canal_System_of_the_State.pdf"
# C:\repos\postgres_ai\ai_test-db\files\02a_Features-ChamplainCanal_final.pdf
# json_file = process_document(pdf_path)



print("\n🎉 Done processing all documents!")
print(f"Total processed: {len(results)}")


🎉 Done processing all documents!
Total processed: 0


<>:15: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:15: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\tinadmin\AppData\Local\Temp\2\ipykernel_15428\4235936960.py:15: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  pdf_path = "C:\\repos\postgres_ai\\ai_test-db\python\\files\\1905 Whitford History_of_the_Canal_System_of_the_State.pdf"


In [22]:
from pathlib import Path
import json

def save_graph_json(pdf_path, final_graph):
    pdf_path = Path(pdf_path)

    # Recommended naming: name.graph.json → safe & consistent
    output_path = pdf_path.with_suffix(pdf_path.suffix + ".graph.json")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_graph, f, indent=2, ensure_ascii=False)

    print(f"✔ Saved graph JSON → {output_path}")
    return output_path


In [23]:
def process_document_for_graph(pdf_path):
    # 1. Extract raw text (OCR or native)
    pages = extract_text_from_pdf(pdf_path)
    text = "\n".join(p["text"] for p in pages)

    # 2. Primary extraction
    raw_graph = extract_relationships(text, aoai)
    save_graph_json(pdf_path + ".raw", raw_graph)

    # 3. Refinement pass
    final_graph = refine_graph(raw_graph, aoai)
    save_graph_json(pdf_path + ".final", final_graph)

   

    return final_graph


In [43]:
process_document_for_graph(pdf_path)



RateLimitError: Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Your requests to gpt-4.1 for gpt-4.1 in East US 2 have exceeded the token rate limit for your current OpenAI S0 pricing tier. This request was for ChatCompletions_Create under Azure OpenAI API version 2024-05-01-preview. Please retry after 60 seconds. To increase your default rate limit, visit: https://aka.ms/oai/quotaincrease.'}}

## For next Demo
* Integrate AI Search Index with RAG
* Show this as Agentic (build agents instead of notebook)
* With User INterface and API 

### Ingest the json file into db

In [37]:
print("Nodes:", len(fetch_nodes()))
print("Edges:", len(fetch_edges()))


Nodes: 178
Edges: 60


In [36]:
# graph_json = process_document(pdf_path)

json_file = "C:\\repos\postgres_ai\\ai_test-db\python\\files\\02a_Features-ChamplainCanal_print.pdf.final.graph.json"

print(json_file)

# if graph_json:
#     ingest_extractions(graph_json)


ingest_json_file(json_file)

<>:3: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:3: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\tinadmin\AppData\Local\Temp\2\ipykernel_15428\1727891129.py:3: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  json_file = "C:\\repos\postgres_ai\\ai_test-db\python\\files\\02a_Features-ChamplainCanal_print.pdf.final.graph.json"


C:\repos\postgres_ai\ai_test-db\python\files\02a_Features-ChamplainCanal_print.pdf.final.graph.json
✔ Graph updated.
✔ Inserted C:\repos\postgres_ai\ai_test-db\python\files\02a_Features-ChamplainCanal_print.pdf.final.graph.json into PostgreSQL


### Run Visualizations

In [87]:
def reset_graph(graph_name="canals_graph"):
    conn = get_conn()
    with conn:
        with conn.cursor() as cur:
            cur.execute("LOAD 'age';")
            cur.execute("SET search_path = ag_catalog, \"$user\", public;")

            # Drop the graph completely
            cur.execute(f"SELECT drop_graph('{graph_name}', true);")

            # Recreate the graph
            cur.execute(f"SELECT create_graph('{graph_name}');")

    conn.close()
    print(f"✔ Graph '{graph_name}' has been reset.")


In [ ]:
reset_graph()

In [ ]:
visualize_graph()


In [34]:
def safe_reset_graph(graph_name="canals_graph"):
    conn = get_conn()
    with conn:
        with conn.cursor() as cur:
            cur.execute("LOAD 'age';")
            cur.execute("SET search_path = ag_catalog, \"$user\", public;")

            # 1. Kill blocking sessions
            cur.execute("""
                SELECT pg_terminate_backend(pid)
                FROM pg_stat_activity
                WHERE state = 'idle in transaction';
            """)

            # 2. Try dropping the graph if it exists
            try:
                cur.execute(f"SELECT drop_graph('{graph_name}', TRUE);")
                print(f"Dropped graph '{graph_name}'.")
            except Exception as e:
                print(f"drop_graph exception (expected if graph missing): {e}")

            # 3. Recreate graph (ensures it exists before cleanup queries)
            try:
                cur.execute(f"SELECT create_graph('{graph_name}');")
                print(f"Created new graph '{graph_name}'.")
            except Exception as e:
                print(f"create_graph error: {e}")
                raise

            # 4. Now that the graph exists, delete any stray nodes (should be empty anyway)
            try:
                cur.execute(f"""
                    SELECT * FROM cypher('{graph_name}', $$
                        MATCH (n) DETACH DELETE n
                    $$) AS (v agtype);
                """)
                print("Cleared nodes (if any).")
            except Exception as e:
                print(f"Cleanup cypher error (graph empty or fresh): {e}")

    print("✔ Safe graph reset complete.\n")


In [35]:
safe_reset_graph()

Dropped graph 'canals_graph'.
Created new graph 'canals_graph'.
Cleared nodes (if any).
✔ Safe graph reset complete.

